# Parametrized Types

Types can be specified as parameters in classes and methods, using the syntax `[T]`.

Before looking into that we can see how we can use these parameters in exsiting generic classes.


### Option types

One example of a type parameter is in generic classes, like `Option`. We cna specify that we want an Option of `Int`:

In [1]:
val maybeName:Option[Int] = Some("ralph")

-- [E007] Type Mismatch Error: cmd1.sc:1:33 ------------------------------------
1 |val maybeName:Option[Int] = Some("ralph")
  |                                 ^^^^^^^
  |                                 Found:    ("ralph" : String)
  |                                 Required: Int
  |
  | longer explanation available when compiling with `-explain`Compilation Failed

: 

This helps restricting the type of data and values that are acceptable.

In [1]:
val maybeName:Option[String] = Some("ralph")

maybeName: Option[String] = Some(value = "ralph")

### Declaring type parameters in classes

We can declare a class where some types are parametrizable:

In [6]:
case class Person[T](name:String,age:T)

defined class Person

Then when we create instances, they can be using different specific types instead of `T`, like `Int` or `Double`.

In [7]:
val jimmy = Person("jimmy",34)

jimmy: Person[Int] = Person(name = "jimmy", age = 34)

In [8]:
val luigi = Person("luigi",34.5)

luigi: Person[Double] = Person(name = "luigi", age = 34.5)

In [9]:
val jenny = Person("jenny","sixteen")

jenny: Person[String] = Person(name = "jenny", age = "sixteen")

The explicit type parameter helps avoiding inconsistencies:

In [9]:
val lara:Person[Int] = Person("lara",15.6)

-- [E007] Type Mismatch Error: cmd9.sc:1:37 ------------------------------------
1 |val lara:Person[Int] = Person("lara",15.6)
  |                                     ^^^^
  |                                     Found:    (15.6d : Double)
  |                                     Required: Int
  |
  | longer explanation available when compiling with `-explain`Compilation Failed

: 

### Type parameters restrictions

However sometimes type parameters are too general and do not always make sense. For example age of a `Student`

In [3]:
case class Student[P](age:P)

defined class Student

We can declare instances with any age type:

In [5]:
val s = Student(true)

s: Student[Boolean] = Student(age = true)

In [6]:
val s = Student(List(2,3,4))

s: Student[List[Int]] = Student(age = List(2, 3, 4))

So these exmaples make little sense. We can restrict the age to only be of value data types:

In [7]:
case class Student[T<:AnyVal](age:T)

defined class Student

In [8]:
val s = Student(List(35,6,3))

-- [E007] Type Mismatch Error: cmd8.sc:1:20 ------------------------------------
1 |val s = Student(List(35,6,3))
  |                ^^^^^^^^^^^^
  |Found:    List[Int]
  |Required: AnyVal
  |Note that implicit conversions cannot be applied because they are ambiguous;
  |both method Ensuring in object Predef and method ArrowAssoc in object Predef convert from List[Int] to T
  |
  | longer explanation available when compiling with `-explain`Compilation Failed

: 

### Parametrized types and inheritance

We can see other effects of type parameters and inheritance. Consider this hierarchy.

In [3]:
trait Athlete

class Swimmer extends Athlete

class Fotballer extends Athlete

defined trait Athlete
defined class Swimmer
defined class Fotballer

We can create a team class with a champion, whose type is parametrizable:

In [4]:
case class SportsTeam[T<:Athlete](champion:T)

defined class SportsTeam

Now if we create instances, the type restrictions prevents form using arbitrary types:

In [4]:
val swimTeam = SportsTeam[String]

-- [E057] Type Mismatch Error: cmd4.sc:1:26 ------------------------------------
1 |val swimTeam = SportsTeam[String]
  |                          ^
  |Type argument String does not conform to upper bound ammonite.$sess.cmd3.wrapper.cmd2.Athlete
  |
  | longer explanation available when compiling with `-explain`Compilation Failed

: 

But it will work for any subtype of `Athlete`:

In [5]:
val alex =  new Swimmer
val swimTeam = SportsTeam[Swimmer](alex)

alex: Swimmer = ammonite.$sess.cmd2$Helper$Swimmer@7d6d5e4f
swimTeam: SportsTeam[Swimmer] = SportsTeam(
  champion = ammonite.$sess.cmd2$Helper$Swimmer@7d6d5e4f
)

The same for a football player:

In [6]:
val jake = new Fotballer
val sionTeam = SportsTeam[Fotballer](jake)

jake: Fotballer = ammonite.$sess.cmd2$Helper$Fotballer@28d22b06
sionTeam: SportsTeam[Fotballer] = SportsTeam(
  champion = ammonite.$sess.cmd2$Helper$Fotballer@28d22b06
)

However, can we cast the sport team with the supertype?

In [6]:
val someTeam:SportsTeam[Athlete] = sionTeam

-- [E007] Type Mismatch Error: cmd6.sc:1:35 ------------------------------------
1 |val someTeam:SportsTeam[Athlete] = sionTeam
  |                                   ^^^^^^^^
  |             Found:    (cmd6.this.cmd5.sionTeam : 
  |               ammonite.$sess.cmd5².wrapper.cmd3.SportsTeam[
  |                 ammonite.$sess.cmd5².wrapper.cmd2.Fotballer
  |               ]
  |             )
  |             Required: cmd6.this.cmd3².SportsTeam[cmd6.this.cmd2².Athlete]
  |
  |             where:    cmd2  is a value in class cmd5
  |                       cmd2² is a value in class cmd6
  |                       cmd3  is a value in class cmd5
  |                       cmd3² is a value in class cmd6
  |                       cmd5  is a value in class cmd6
  |                       cmd5² is a object in package ammonite.$sess
  |
  | longer explanation available when compiling with `-explain`Compilation Failed

: 

### Covariants

As it was defined, it is an invariant, so the `SportsTeam[Footballer]` is not recognized as subtype of `SportsTeam[Athlete]`.
To fix this we can use covariants, defined using `+` before the type parameter.

In [7]:
case class SportsTeam[+T<:Athlete](champion:T)

defined class SportsTeam

Now the intuitive subtyping is working correctly:

In [8]:
val jake = new Fotballer
val sionTeam = SportsTeam[Fotballer](jake)
val someTeam:SportsTeam[Athlete] = sionTeam

jake: Fotballer = ammonite.$sess.cmd2$Helper$Fotballer@299056c0
sionTeam: SportsTeam[Fotballer] = SportsTeam(
  champion = ammonite.$sess.cmd2$Helper$Fotballer@299056c0
)
someTeam: SportsTeam[Athlete] = SportsTeam(
  champion = ammonite.$sess.cmd2$Helper$Fotballer@299056c0
)

### Type params in methods

We can also define type parameters in methods.

In [9]:
class SportsCompetition[T]


def giveAward[T](competition:SportsCompetition[T], athlete:T) = println ("winner is "+athlete)

defined class SportsCompetition
defined function giveAward

The the type of competition can be defined when using the method.

In [10]:
val swissCup = new SportsCompetition[Fotballer]

val shakiri = new Fotballer

giveAward(swissCup,shakiri)

winner is ammonite.$sess.cmd2$Helper$Fotballer@17a6dfc0


swissCup: SportsCompetition[Fotballer] = ammonite.$sess.cmd8$Helper$SportsCompetition@6e6f3849
shakiri: Fotballer = ammonite.$sess.cmd2$Helper$Fotballer@17a6dfc0

What if we do the same, but in this case the competition is for any athelete?

In [10]:
val swissCup = new SportsCompetition[Athlete]

val shakiri = new Fotballer

giveAward[Fotballer](swissCup,shakiri)

-- [E007] Type Mismatch Error: cmd10.sc:5:35 -----------------------------------
5 |val res10_2 = giveAward[Fotballer](swissCup,shakiri)
  |                                   ^^^^^^^^
  |    Found:    (Helper.this.swissCup : 
  |      cmd10.this.cmd8.SportsCompetition[cmd10.this.cmd2.Athlete]
  |    )
  |    Required: cmd10.this.cmd8.SportsCompetition[cmd10.this.cmd2.Fotballer]
  |
  | longer explanation available when compiling with `-explain`Compilation Failed

: 

### Contravariance

As opposed to covariants, we can solve this problem with contravariants, which are the opposite. We can use the supertype as parameter.
The notation for contravariants is `-`


In [11]:
class SportsCompetition[-T]


def giveAward[T](competition:SportsCompetition[T], athlete:T) = println ("winner is "+athlete)

defined class SportsCompetition
defined function giveAward

Now the award of a Footballer to an Athlete competition is possible.

In [12]:
val swissCup = new SportsCompetition[Athlete]

val shakiri = new Fotballer

giveAward[Fotballer](swissCup,shakiri)

winner is ammonite.$sess.cmd2$Helper$Fotballer@69f93bb


swissCup: SportsCompetition[Athlete] = ammonite.$sess.cmd10$Helper$SportsCompetition@13df07f3
shakiri: Fotballer = ammonite.$sess.cmd2$Helper$Fotballer@69f93bb